# StatsPAI Causal Inference Demo — Interactive Edition

This notebook demonstrates core causal inference methods with **interactive plot editing**.

Every plot opens with `sp.interactive(fig)` — an ipywidgets control panel where you can:
- Edit titles, labels, fonts, colors, line styles
- Adjust DPI, figure size, tick rotation, background
- Add annotations, toggle grid/legend/spines
- Generate reproducible code for all edits

**Methods covered:**

1. **Difference-in-Differences (DID)** — Classic 2×2 and Event Study
2. **Fixed Effects (FE)** — Panel data with entity fixed effects
3. **Instrumental Variables (IV)** — 2SLS estimation
4. **Matching** — Propensity score matching
5. **Synthetic Control (SC)** — Abadie-style SCM
6. **Regression Discontinuity (RDD)** — Sharp RD design
7. **Binscatter** — Non-parametric conditional means

---

# StatsPAI 因果推断演示 — 交互式版本

本 Notebook 展示了核心因果推断方法，并支持**交互式图表编辑**。

每张图表都通过 `sp.interactive(fig)` 打开一个 ipywidgets 控制面板，您可以：
- 编辑标题、标签、字体、颜色、线条样式
- 调整 DPI、图形大小、刻度旋转、背景色
- 添加注释，切换网格/图例/边框
- 生成所有编辑的可复现代码

**涵盖的方法：**

1. **双重差分法 (DID)** — 经典 2×2 设计与事件研究
2. **固定效应模型 (FE)** — 面板数据实体固定效应
3. **工具变量法 (IV)** — 两阶段最小二乘估计
4. **匹配法 (Matching)** — 倾向得分匹配
5. **合成控制法 (SC)** — Abadie 式合成控制
6. **断点回归设计 (RDD)** — 精确断点回归
7. **分箱散点图 (Binscatter)** — 非参数条件均值可视化

In [1]:
# 导入依赖库并设置随机种子和学术主题
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statspai as sp

np.random.seed(42)
sp.set_theme('academic')
%matplotlib inline

---
## 1. Difference-in-Differences (DID)

Classic 2×2 DID with parallel trends visualization, then staggered adoption event study.

---
## 1. 双重差分法 (DID)

经典 2×2 双重差分设计，包含平行趋势可视化和交错处理事件研究。

In [2]:
# 模拟 DID 数据：2×2 设计
# 200 个个体，10 个时间段，处理组从 t=6 开始接受处理
# Simulate DID data: 2×2 design
# 200 individuals, 10 time periods, treatment starts at t=6 for treated group
n_did = 200
t_periods = 10
ids = np.repeat(np.arange(n_did), t_periods)
times = np.tile(np.arange(1, t_periods + 1), n_did)
treat_group = (ids < 100).astype(int)  # first 100 = treated
post = (times >= 6).astype(int)

# Parallel trends in pre-period, treatment effect = 3.0 in post
individual_fe = np.repeat(np.random.normal(0, 2, n_did), t_periods)
time_trend = 0.5 * times
y_did = 10 + 2 * treat_group + time_trend + 3.0 * treat_group * post + individual_fe + np.random.normal(0, 1.5, n_did * t_periods)

df_did = pd.DataFrame({
    'id': ids, 'time': times, 'treated': treat_group,
    'post': post, 'outcome': y_did,
    'x1': np.random.normal(0, 1, n_did * t_periods),
})
print(f"Shape: {df_did.shape}, Treated units: {(ids < 100).sum() // t_periods}")
df_did.head()

Shape: (2000, 6), Treated units: 100


,id,time,treated,post,outcome,x1
0,0,1,1,0,14.030109,1.804348
1,0,2,1,0,14.834605,-0.190904
2,0,3,1,0,16.118005,0.719758
3,0,4,1,0,16.574131,-1.293273
4,0,5,1,0,13.426924,-0.956436


In [3]:
# 2×2 双重差分估计
# 2×2 DID estimation
did_result = sp.did(
    df_did, y='outcome', treat='treated', time='time',
    id='id', method='classic'
)
did_result

variable,coefficient,se,tstat,pvalue
const,11.5861,0.1144,101.2643,0.0000
treated,1.8156,0.1586,11.4494,0.0000
_post,2.5099,0.1601,15.6771,0.0000
treatedx_post,2.9340,0.2251,13.0370,0.0000


In [4]:
# 绘图 1：DID 平行趋势 — 处理组与控制组各期均值
# Plot 1: DID parallel trends — treated vs control group means over time
fig, ax = sp.did_plot(
    df_did, y='outcome', time='time', treat='treated',
    treat_time=6,
    labels={'treated': 'Treatment Group', 'control': 'Control Group'},
    title='DID: Parallel Trends & Treatment Effect',
    annotate_effect=True,
)
# # --- StatsPAI interactive edits (paste after your plot code) ---
# import statspai as sp
# sp.set_theme('ggplot')
# fig.tight_layout()
# # --- end StatsPAI edits ---

sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5052208016: <ArtistRole.LABEL: 5>, 5070435024: <ArtistRole.LABEL: 5>, 5070436432: <ArtistRole.LABEL: 5>, 5051343120: <ArtistRole.SPINE: 8>, 5070430736: <ArtistRole.SPINE: 8>, 5070431952: <ArtistRole.SPINE: 8>, 5070433104: <ArtistRole.SPINE: 8>, 5051339664: <ArtistRole.LEGEND: 7>, 5051403216: <ArtistRole.DATA: 1>, 5070688016: <ArtistRole.REFERENCE: 4>, 5070693008: <ArtistRole.DATA: 1>, 5051346320: <ArtistRole.REFERENCE: 4>, 5070699088: <ArtistRole.ANNOTATION: 6>, 5070525264: <ArtistRole.ANNOTATION: 6>, 4464028688: <ArtistRole.ANNOTATION: 6>}, _original_state={'ax.title.text': 'DID: Parallel Trends & Treatment Effect', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Time', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'Outcome', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(0.

In [5]:
# 绘图 2：使用 Callaway-Sant'Anna 方法的事件研究
# 需要交错处理编码：first_treat = 首次处理时期（0 = 从未处理）
# Plot 2: Event study using Callaway-Sant'Anna
# Need staggered treatment coding: first_treat = period of first treatment (0 = never treated)
df_did['first_treat'] = np.where(df_did['treated'] == 1, 6, 0)

es_result = sp.did(
    df_did, y='outcome', treat='first_treat', time='time',
    id='id', method='cs'
)
fig, ax = es_result.plot(type='event_study',
    title='DID Event Study: Dynamic Treatment Effects',
    color='#1a5276',
)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5077426960: <ArtistRole.LABEL: 5>, 5078185552: <ArtistRole.LABEL: 5>, 4387152592: <ArtistRole.LABEL: 5>, 5077950224: <ArtistRole.SPINE: 8>, 5078177744: <ArtistRole.SPINE: 8>, 5077299728: <ArtistRole.SPINE: 8>, 5077295056: <ArtistRole.SPINE: 8>, 5078690576: <ArtistRole.LEGEND: 7>, 5076699408: <ArtistRole.DATA: 1>, 5037194448: <ArtistRole.REFERENCE: 4>, 5078501840: <ArtistRole.REFERENCE: 4>, 5078066576: <ArtistRole.REFERENCE: 4>, 5078686992: <ArtistRole.REFERENCE: 4>, 5078488592: <ArtistRole.CI: 3>, 5077121232: <ArtistRole.DATA: 1>, 5077117712: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'DID Event Study: Dynamic Treatment Effects', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Periods Relative to Treatment', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'Estimated Effect', 'ax.ylabel.fontsize': 

---
## 2. Fixed Effects (FE)

Panel data: 100 firms observed over 10 years. Treatment `policy` is staggered.

---
## 2. 固定效应模型 (FE)

面板数据：100 家企业观测 10 年。处理变量 `policy`（政策）采用交错实施。

In [6]:
# 模拟面板数据：100 家企业，10 年
# Simulate panel data
n_firms, n_years = 100, 10
firms = np.repeat(np.arange(n_firms), n_years)
years = np.tile(np.arange(2010, 2010 + n_years), n_firms)
firm_fe = np.repeat(np.random.normal(0, 2, n_firms), n_years)
x1 = np.random.normal(5, 1, n_firms * n_years)
x2 = np.random.normal(0, 1, n_firms * n_years)
policy = ((years >= 2015) & (firms < 50)).astype(int)
y_panel = 3 + 2.5 * x1 - 1.2 * x2 + 1.8 * policy + firm_fe + np.random.normal(0, 1, n_firms * n_years)

df_panel = pd.DataFrame({
    'firm_id': firms, 'year': years,
    'revenue': y_panel, 'investment': x1, 'debt_ratio': x2, 'policy': policy
})
df_panel.head()

,firm_id,year,revenue,investment,debt_ratio,policy
0,0,2010,14.957120,4.576401,-0.208668,0
1,0,2011,11.507491,3.634383,-0.312668,0
2,0,2012,13.481348,4.025878,0.857899,0
3,0,2013,18.209287,5.703428,0.553570,0
4,0,2014,11.080308,3.692630,-0.257300,0


In [7]:
# 固定效应回归
# Fixed Effects regression
fe_result = sp.panel(
    df_panel,
    "revenue ~ investment + debt_ratio + policy",
    entity='firm_id', time='year', method='fe'
)
fe_result

,Coefficient,Std. Error,t-stat,P>|t|,95% CI
investment,2.5327 ***,(0.0352),71.88,0.0000,"[2.464, 2.602]"
debt_ratio,-1.1681 ***,(0.0343),-34.10,0.0000,"[-1.235, -1.101]"
policy,2.0139 ***,(0.0931),21.64,0.0000,"[1.831, 2.197]"


In [8]:
# Hausman 检验：固定效应 vs 随机效应
# Hausman test: FE vs RE
fe_result.hausman_test()

{'statistic': 0.29638622861747155,
 'df': 3,
 'pvalue': 0.9607066942780254,
 'recommendation': 'RE',
 'beta_fe': investment    2.532736
 debt_ratio   -1.168082
 policy        2.013911
 dtype: float64,
 'beta_re': investment    2.532659
 debt_ratio   -1.168357
 policy        2.009315
 dtype: float64,
 'interpretation': 'chi2(3) = 0.2964, p = 0.9607. Cannot reject H0: Random Effects is more efficient.'}

In [9]:
# 绘图：系数森林图
# Plot: Coefficient forest plot
fig, ax = sp.coefplot(fe_result, title='Fixed Effects: Coefficient Plot')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5095809168: <ArtistRole.LABEL: 5>, 5095800976: <ArtistRole.LABEL: 5>, 5095803344: <ArtistRole.LABEL: 5>, 5079178448: <ArtistRole.SPINE: 8>, 5095723728: <ArtistRole.SPINE: 8>, 5095732368: <ArtistRole.SPINE: 8>, 5095733520: <ArtistRole.SPINE: 8>, 5095804624: <ArtistRole.LEGEND: 7>, 5095651280: <ArtistRole.REFERENCE: 4>, 5095843728: <ArtistRole.REFERENCE: 4>, 5095596048: <ArtistRole.REFERENCE: 4>, 5095602384: <ArtistRole.DATA: 1>, 5095721168: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'Fixed Effects: Coefficient Plot', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Coefficient Estimate', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': '', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(-1.4270658370199154), np.float64(2.7936456216609726)), 'ax.ylim': (np.float64(2.1), 

In [10]:
# 绘图：对比固定效应、随机效应和混合回归的系数
# Plot: Compare FE vs RE vs Pooled coefficients
re_result = sp.panel(df_panel, "revenue ~ investment + debt_ratio + policy",
                     entity='firm_id', time='year', method='re')
pooled_result = sp.panel(df_panel, "revenue ~ investment + debt_ratio + policy",
                         entity='firm_id', time='year', method='pooled')

fig, ax = sp.coefplot(fe_result, re_result, pooled_result,
                      model_names=['FE', 'RE', 'Pooled'],
                      title='Panel Models: Coefficient Comparison')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5096740752: <ArtistRole.LABEL: 5>, 5097670160: <ArtistRole.LABEL: 5>, 5097670992: <ArtistRole.LABEL: 5>, 5097762576: <ArtistRole.SPINE: 8>, 5097402256: <ArtistRole.SPINE: 8>, 5096618704: <ArtistRole.SPINE: 8>, 5096619856: <ArtistRole.SPINE: 8>, 5097672400: <ArtistRole.LEGEND: 7>, 5097762768: <ArtistRole.REFERENCE: 4>, 5097837840: <ArtistRole.REFERENCE: 4>, 5097845648: <ArtistRole.REFERENCE: 4>, 5097918288: <ArtistRole.REFERENCE: 4>, 5096865104: <ArtistRole.REFERENCE: 4>, 5096274320: <ArtistRole.REFERENCE: 4>, 5097760016: <ArtistRole.REFERENCE: 4>, 5096931152: <ArtistRole.DATA: 1>, 5097807888: <ArtistRole.CI: 3>, 5097836688: <ArtistRole.DATA: 1>, 5096606480: <ArtistRole.CI: 3>, 5097923408: <ArtistRole.DATA: 1>, 5097848528: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'Panel Models: Coefficient Comparison', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 

---
## 3. Instrumental Variables (IV)

Classic returns-to-education example: `education` is endogenous, `distance_to_college` is the instrument.

---
## 3. 工具变量法 (IV)

经典的教育回报率案例：`education`（教育年限）为内生变量，`distance_to_college`（到大学的距离）为工具变量。

In [11]:
# 模拟工具变量数据
# ability（能力）为不可观测变量，distance（距离）为工具变量
# Simulate IV data
n_iv = 2000
ability = np.random.normal(0, 1, n_iv)  # unobserved
distance = np.random.normal(50, 20, n_iv)  # instrument
experience = np.random.normal(10, 3, n_iv)
education = 16 - 0.08 * distance + 0.5 * ability + np.random.normal(0, 1, n_iv)
wage = 10 + 0.8 * education + 0.3 * experience + 0.6 * ability + np.random.normal(0, 2, n_iv)

df_iv = pd.DataFrame({
    'wage': wage, 'education': education,
    'experience': experience, 'distance': distance
})
df_iv.head()

,wage,education,experience,distance
0,16.860593,11.152590,10.657927,58.424948
1,23.263697,10.456267,10.105886,66.936799
2,21.417441,10.372786,7.831496,60.686944
3,23.347924,15.824814,3.473354,14.631691
4,20.524002,8.772993,7.021394,69.903353


In [12]:
# OLS 回归（因遗漏能力变量而存在偏误）
# OLS (biased due to omitted ability)
ols_result = sp.regress("wage ~ education + experience", data=df_iv)
ols_result

,Coefficient,Std. Error,t-stat,P>|t|,95% CI
Intercept,9.0207 ***,(0.3337),27.03,0.0000,"[8.366, 9.675]"
education,0.9003 ***,(0.0241),37.31,0.0000,"[0.853, 0.948]"
experience,0.2823 ***,(0.0155),18.17,0.0000,"[0.252, 0.313]"


In [13]:
# 两阶段最小二乘 (2SLS) 工具变量估计
# 公式：y ~ (内生变量 ~ 工具变量) + 外生控制变量
# 2SLS IV estimation
# Formula: y ~ (endogenous ~ instruments) + exogenous_controls
iv_result = sp.iv(
    "wage ~ (education ~ distance) + experience",
    data=df_iv
)
iv_result

,Coefficient,Std. Error,t-stat,P>|t|,95% CI
Intercept,10.0423 ***,(0.3978),25.24,0.0000,"[9.262, 10.822]"
experience,0.2822 ***,(0.0156),18.12,0.0000,"[0.252, 0.313]"
education,0.8157 ***,(0.0301),27.14,0.0000,"[0.757, 0.875]"


In [14]:
# 绘图：并排比较 OLS 与 IV 系数
# Plot: Compare OLS vs IV coefficients side by side
fig, ax = sp.coefplot(ols_result, iv_result,
                      model_names=['OLS (biased)', '2SLS IV (consistent)'],
                      colors=['#E74C3C', '#2C3E50'],
                      title='OLS vs IV: Returns to Education')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5282398544: <ArtistRole.LABEL: 5>, 5282371664: <ArtistRole.LABEL: 5>, 5282374608: <ArtistRole.LABEL: 5>, 5282365712: <ArtistRole.SPINE: 8>, 5282367120: <ArtistRole.SPINE: 8>, 5282368464: <ArtistRole.SPINE: 8>, 5282369616: <ArtistRole.SPINE: 8>, 5282667088: <ArtistRole.LEGEND: 7>, 5098865808: <ArtistRole.REFERENCE: 4>, 5282633680: <ArtistRole.REFERENCE: 4>, 5282641680: <ArtistRole.REFERENCE: 4>, 5282668432: <ArtistRole.REFERENCE: 4>, 5282641744: <ArtistRole.REFERENCE: 4>, 5282557392: <ArtistRole.DATA: 1>, 5282560208: <ArtistRole.CI: 3>, 5282639696: <ArtistRole.DATA: 1>, 5282305104: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'OLS vs IV: Returns to Education', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Coefficient Estimate', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': '', 'ax.ylabel.fontsize

---
## 4. Matching

Job training program evaluation: estimate ATT using propensity score matching.

---
## 4. 匹配法 (Matching)

就业培训项目评估：使用倾向得分匹配估计处理组平均处理效应（ATT）。

In [15]:
# 模拟匹配数据：就业培训项目
# Simulate matching data
n_match = 1000
age = np.random.normal(35, 8, n_match)
edu_years = np.random.normal(12, 3, n_match)
prior_earn = np.random.normal(30000, 8000, n_match)

# Treatment assignment depends on covariates (selection bias)
propensity = 1 / (1 + np.exp(-(- 2 + 0.05 * age - 0.1 * edu_years + 0.00005 * prior_earn)))
treatment = np.random.binomial(1, propensity)

# Outcome: treatment effect = 2000
earnings = 25000 + 500 * edu_years + 100 * age + 0.5 * prior_earn + 2000 * treatment + np.random.normal(0, 3000, n_match)

df_match = pd.DataFrame({
    'earnings': earnings, 'treatment': treatment,
    'age': age, 'edu_years': edu_years, 'prior_earn': prior_earn
})
print(f"Treated: {treatment.sum()}, Control: {(1-treatment).sum()}")
df_match.head()

Treated: 485, Control: 515


,earnings,treatment,age,edu_years,prior_earn
0,52592.516393,1,20.702891,13.150965,41327.285124
1,51203.815118,1,26.931754,9.365591,38008.518741
2,45276.029806,0,33.025137,8.042484,32374.616868
3,51692.670654,0,22.516532,11.528465,28293.692693
4,56786.043023,0,43.897193,10.427315,36819.271022


In [16]:
# 倾向得分匹配
# Propensity Score Matching
match_result = sp.match(
    df_match,
    y='earnings', treat='treatment',
    covariates=['age', 'edu_years', 'prior_earn'],
    distance='propensity', method='nearest',
    n_matches=3, caliper=0.2
)
match_result

/var/folders/bm/2cq1vtmd5wg6k0vdjfpvbmww0000gn/T/ipykernel_15847/717465332.py:3: UserWarning: PSM can increase imbalance and bias (King & Nielsen 2019). Consider distance='mahalanobis' or method='cem'.
  match_result = sp.match(


Variable,Treated,Control,SMD,
age,36.83,33.63,0.401,
edu_years,11.59,12.38,-0.268,
prior_earn,31168.28,28972.14,0.273,
propensity_score,0.53,0.45,0.584,


In [17]:
# 绘图：Love 图（匹配前后协变量平衡性）
# Plot: Love plot (covariate balance before/after matching)
fig, ax = sp.balanceplot(match_result, title='Covariate Balance: Love Plot')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x600 with 1 Axes>, protect_data=True, edits=[], artist_roles={5283373712: <ArtistRole.LABEL: 5>, 5283853904: <ArtistRole.LABEL: 5>, 5283099216: <ArtistRole.LABEL: 5>, 5283487504: <ArtistRole.SPINE: 8>, 5282663248: <ArtistRole.SPINE: 8>, 5282961680: <ArtistRole.SPINE: 8>, 5282966608: <ArtistRole.SPINE: 8>, 5283995088: <ArtistRole.REFERENCE: 4>, 5283410128: <ArtistRole.REFERENCE: 4>, 5283187408: <ArtistRole.REFERENCE: 4>, 5283035344: <ArtistRole.DATA: 1>, 5283401616: <ArtistRole.COSMETIC: 9>, 5283811216: <ArtistRole.COSMETIC: 9>, 5283404816: <ArtistRole.COSMETIC: 9>, 5284094416: <ArtistRole.COSMETIC: 9>}, _original_state={'ax.title.text': 'Covariate Balance: Love Plot', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Standardized Mean Difference (SMD)', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': '', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (

In [18]:
# 绘图：倾向得分分布（共同支撑域）
# Plot: Propensity score distribution (common support)
fig, ax = sp.psplot(
    df_match, treat='treatment',
    covariates=['age', 'edu_years', 'prior_earn'],
    title='Propensity Score Distribution',
    colors=('#3498DB', '#E74C3C'),
    labels=('Control Group', 'Training Program'),
)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x750 with 1 Axes>, protect_data=True, edits=[], artist_roles={5285065936: <ArtistRole.LABEL: 5>, 5283028688: <ArtistRole.LABEL: 5>, 5284472080: <ArtistRole.LABEL: 5>, 5284618896: <ArtistRole.SPINE: 8>, 5284618064: <ArtistRole.SPINE: 8>, 5283278032: <ArtistRole.SPINE: 8>, 5284404560: <ArtistRole.SPINE: 8>, 5284944336: <ArtistRole.LEGEND: 7>, 5284143696: <ArtistRole.COSMETIC: 9>, 5284530960: <ArtistRole.COSMETIC: 9>, 5285720080: <ArtistRole.COSMETIC: 9>, 5098868944: <ArtistRole.COSMETIC: 9>, 5284692496: <ArtistRole.COSMETIC: 9>, 5284695056: <ArtistRole.COSMETIC: 9>, 5284697808: <ArtistRole.COSMETIC: 9>, 5284700368: <ArtistRole.COSMETIC: 9>, 5284692432: <ArtistRole.COSMETIC: 9>, 5284694992: <ArtistRole.COSMETIC: 9>, 5284707408: <ArtistRole.COSMETIC: 9>, 5285693008: <ArtistRole.COSMETIC: 9>, 5285695376: <ArtistRole.COSMETIC: 9>, 5285697936: <ArtistRole.COSMETIC: 9>, 5285700240: <ArtistRole.COSMETIC: 9>, 5283803152: <ArtistRole.COSMETIC: 9>, 5285704912: <Ar

---
## 5. Synthetic Control (SC)

A policy intervention in State 1 starting in year 2005. 10 donor states.

---
## 5. 合成控制法 (SC)

州 1 在 2005 年实施政策干预，使用 10 个捐赠州构建合成控制。

In [19]:
# 模拟合成控制数据：11 个州，20 个时期
# Simulate synthetic control data
n_states, n_periods = 11, 20
states = np.repeat(np.arange(1, n_states + 1), n_periods)
time_periods = np.tile(np.arange(2000, 2000 + n_periods), n_states)

# Each state has a base trend + state-specific intercept
state_intercepts = np.repeat(np.random.normal(50, 5, n_states), n_periods)
common_trend = np.tile(np.linspace(0, 10, n_periods), n_states)
noise = np.random.normal(0, 1, n_states * n_periods)

gdp = state_intercepts + common_trend + noise

# Add treatment effect for state 1 after 2005 (effect = +5)
treatment_effect = 5.0 * ((states == 1) & (time_periods >= 2005)).astype(float)
gdp += treatment_effect

df_synth = pd.DataFrame({
    'state': states, 'year': time_periods, 'gdp': gdp
})
df_synth.head()

,state,year,gdp
0,1,2000,47.983380
1,1,2001,49.634079
2,1,2002,48.875418
3,1,2003,50.008466
4,1,2004,49.088300


In [20]:
# 经典合成控制法
# Classic Synthetic Control
sc_result = sp.synth(
    df_synth,
    outcome='gdp', unit='state', time='year',
    treated_unit=1, treatment_time=2005
)
sc_result

time,treated,synthetic,gap,post_treatment
2000,47.9834,48.4940,-0.5106,False
2001,49.6341,48.6727,0.9614,False
2002,48.8754,49.5658,-0.6903,False
2003,50.0085,49.4085,0.6000,False
2004,49.0883,49.8450,-0.7567,False
2005,56.6976,51.2974,5.4001,True
2006,55.2257,50.8036,4.4220,True
2007,56.3369,51.5942,4.7427,True
2008,59.0126,51.5605,7.4521,True
2009,59.5021,53.1697,6.3324,True


In [21]:
# 绘图：处理单元 vs 合成控制轨迹
# Plot: Treated vs Synthetic trajectory
fig, ax = sp.synthplot(sc_result, type='trajectory',
                       title='Synthetic Control: State 1 vs Synthetic')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x975 with 1 Axes>, protect_data=True, edits=[], artist_roles={5287207824: <ArtistRole.LABEL: 5>, 5286622800: <ArtistRole.LABEL: 5>, 5287458192: <ArtistRole.LABEL: 5>, 5286568144: <ArtistRole.SPINE: 8>, 5287232976: <ArtistRole.SPINE: 8>, 5287337104: <ArtistRole.SPINE: 8>, 5287331152: <ArtistRole.SPINE: 8>, 5285796240: <ArtistRole.LEGEND: 7>, 5285522960: <ArtistRole.DATA: 1>, 5286738128: <ArtistRole.DATA: 1>, 5286462864: <ArtistRole.REFERENCE: 4>, 5287150800: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'Synthetic Control: State 1 vs Synthetic', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Time', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'Outcome', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(1999.05), np.float64(2019.95)), 'ax.ylim': (np.float64(47.12487078585494), np.float64(66.01208356282729)), 'ax.facecolor': (1.0, 

In [22]:
# 绘图：处理效应差距随时间变化
# Plot: Treatment effect gap over time
fig, ax = sp.synthplot(sc_result, type='gap',
                       title='SC: Treatment Effect Gap (Treated - Synthetic)')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x750 with 1 Axes>, protect_data=True, edits=[], artist_roles={5288567568: <ArtistRole.LABEL: 5>, 5288395280: <ArtistRole.LABEL: 5>, 5288365392: <ArtistRole.LABEL: 5>, 5288490320: <ArtistRole.SPINE: 8>, 5288484944: <ArtistRole.SPINE: 8>, 5287573712: <ArtistRole.SPINE: 8>, 5288404304: <ArtistRole.SPINE: 8>, 5288814928: <ArtistRole.DATA: 1>, 5287848400: <ArtistRole.REFERENCE: 4>, 5288610832: <ArtistRole.REFERENCE: 4>, 5288297680: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'SC: Treatment Effect Gap (Treated - Synthetic)', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Time', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'Gap (Treated − Synthetic)', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(1999.05), np.float64(2019.95)), 'ax.ylim': (np.float64(-1.1744815757430636), np.float64(8.016774002802455)), 'ax.facecolor': (1.0, 1.0,

In [23]:
# 绘图：捐赠州权重
# Plot: Donor weights
fig, ax = sp.synthplot(sc_result, type='weights',
                       title='SC: Donor State Weights')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1200x750 with 1 Axes>, protect_data=True, edits=[], artist_roles={5289344784: <ArtistRole.LABEL: 5>, 5289670224: <ArtistRole.LABEL: 5>, 5290024976: <ArtistRole.LABEL: 5>, 5289336592: <ArtistRole.SPINE: 8>, 5289202896: <ArtistRole.SPINE: 8>, 5289633360: <ArtistRole.SPINE: 8>, 5289712656: <ArtistRole.SPINE: 8>, 5289351952: <ArtistRole.REFERENCE: 4>, 5290120912: <ArtistRole.COSMETIC: 9>, 5289486352: <ArtistRole.COSMETIC: 9>, 5289845520: <ArtistRole.COSMETIC: 9>, 5290169552: <ArtistRole.COSMETIC: 9>}, _original_state={'ax.title.text': 'SC: Donor State Weights', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Weight', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': '', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(0.0), np.float64(0.4642936592746817)), 'ax.ylim': (np.float64(-0.5900000000000001), np.float64(3.5900000000000003)), 'ax.facecolor': (

In [24]:
# 绘图：安慰剂检验（空间维度）
# Plot: Placebo test (in-space)
fig, ax = sp.synthplot(sc_result, type='placebo',
                       title='SC: In-Space Placebo Test')
sp.interactive(fig)

FigureEditor(fig=<Figure size 1350x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5290630352: <ArtistRole.LABEL: 5>, 5290909648: <ArtistRole.LABEL: 5>, 5290899792: <ArtistRole.LABEL: 5>, 5291064848: <ArtistRole.SPINE: 8>, 5291056336: <ArtistRole.SPINE: 8>, 5290222864: <ArtistRole.SPINE: 8>, 5290910352: <ArtistRole.SPINE: 8>, 5289630160: <ArtistRole.LEGEND: 7>, 5287871824: <ArtistRole.REFERENCE: 4>, 5290289296: <ArtistRole.ANNOTATION: 6>, 5283434576: <ArtistRole.COSMETIC: 9>, 5304101008: <ArtistRole.COSMETIC: 9>, 5304201424: <ArtistRole.COSMETIC: 9>, 5304203792: <ArtistRole.COSMETIC: 9>, 5304009488: <ArtistRole.COSMETIC: 9>, 5304011984: <ArtistRole.COSMETIC: 9>, 5304014864: <ArtistRole.COSMETIC: 9>, 5304017232: <ArtistRole.COSMETIC: 9>}, _original_state={'ax.title.text': 'SC: In-Space Placebo Test', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'ATT', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylab

---
## 6. Regression Discontinuity Design (RDD)

Scholarship eligibility based on a test score cutoff at 60.

---
## 6. 断点回归设计 (RDD)

基于考试分数 60 分断点的奖学金资格评估。

In [25]:
# 模拟 RDD 数据：分数断点在 60 分
# Simulate RDD data
n_rd = 2000
score = np.random.normal(60, 15, n_rd)  # running variable, cutoff at 60
above_cutoff = (score >= 60).astype(int)

# Outcome: GPA with a jump at the cutoff (treatment effect = 0.4)
gpa = 2.5 + 0.02 * score + 0.4 * above_cutoff + np.random.normal(0, 0.3, n_rd)

df_rd = pd.DataFrame({
    'gpa': gpa, 'test_score': score, 'scholarship': above_cutoff
})
df_rd.head()

,gpa,test_score,scholarship
0,4.704528,79.379964,1
1,3.760986,53.293551,0
2,4.232309,83.322604,1
3,4.320970,62.941639,1
4,4.392721,90.379263,1


In [26]:
# 精确断点回归估计
# Sharp RD estimation
rd_result = sp.rdrobust(
    df_rd, y='gpa', x='test_score', c=60
)
rd_result

Method,Estimate,Std. Err.,z,p-value,CI
Conventional,0.4052 ***,(0.0812),4.99,0.0000,"[0.2461, 0.5643]"
Robust,0.3869 ***,(0.1125),3.44,0.0006,"[0.1665, 0.6074]"


In [27]:
# 绘图：RD 散点图（含多项式拟合和置信区间阴影）
# Plot: RD scatter with polynomial fit and CI shading
fig, ax = sp.rdplot(
    df_rd, y='gpa', x='test_score', c=60,
    nbins=40, ci_level=0.95, shade_ci=True,
    show_bw=True,
    title='RDD: Scholarship Effect on GPA',
)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x1050 with 1 Axes>, protect_data=True, edits=[], artist_roles={5305302800: <ArtistRole.LABEL: 5>, 5304055376: <ArtistRole.LABEL: 5>, 5304807440: <ArtistRole.LABEL: 5>, 5304833488: <ArtistRole.SPINE: 8>, 5305089040: <ArtistRole.SPINE: 8>, 5305080208: <ArtistRole.SPINE: 8>, 5290330960: <ArtistRole.SPINE: 8>, 5304753360: <ArtistRole.LEGEND: 7>, 5305561872: <ArtistRole.DATA: 1>, 5305564880: <ArtistRole.DATA: 1>, 5305563024: <ArtistRole.REFERENCE: 4>, 5305542288: <ArtistRole.CI: 3>, 5303877904: <ArtistRole.CI: 3>, 5304934288: <ArtistRole.DATA: 1>, 5305556880: <ArtistRole.DATA: 1>, 5304627984: <ArtistRole.COSMETIC: 9>}, _original_state={'ax.title.text': 'RDD: Scholarship Effect on GPA', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'test_score', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'gpa', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float

In [28]:
# 绘图：密度检验（McCrary 式）— 检查是否存在操纵
# Plot: Density test (McCrary-style) — check for manipulation
fig, ax = sp.rdplotdensity(
    df_rd, x='test_score', c=60,
    hist=True, nbins=40, ci_level=0.95,
    title='RDD: Running Variable Density at Cutoff',
)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x1050 with 1 Axes>, protect_data=True, edits=[], artist_roles={5305454544: <ArtistRole.LABEL: 5>, 5306121744: <ArtistRole.LABEL: 5>, 5306106256: <ArtistRole.LABEL: 5>, 5306237072: <ArtistRole.SPINE: 8>, 5306219664: <ArtistRole.SPINE: 8>, 5306212240: <ArtistRole.SPINE: 8>, 5306206416: <ArtistRole.SPINE: 8>, 5307361744: <ArtistRole.LEGEND: 7>, 5307208400: <ArtistRole.REFERENCE: 4>, 5307521168: <ArtistRole.REFERENCE: 4>, 5290412880: <ArtistRole.REFERENCE: 4>, 5307521616: <ArtistRole.CI: 3>, 5307527248: <ArtistRole.CI: 3>, 5306514512: <ArtistRole.COSMETIC: 9>, 5305995152: <ArtistRole.COSMETIC: 9>, 5306763472: <ArtistRole.COSMETIC: 9>, 5290837840: <ArtistRole.COSMETIC: 9>, 5306768080: <ArtistRole.COSMETIC: 9>, 5306770512: <ArtistRole.COSMETIC: 9>, 5306562320: <ArtistRole.COSMETIC: 9>, 5305946960: <ArtistRole.COSMETIC: 9>, 5306768016: <ArtistRole.COSMETIC: 9>, 5306770448: <ArtistRole.COSMETIC: 9>, 5307027920: <ArtistRole.COSMETIC: 9>, 5306769360: <ArtistRole

In [29]:
# 绘图：带宽敏感性 — 不同带宽下 RD 估计的稳健性
# Plot: Bandwidth sensitivity — RD estimate robustness across bandwidths
_ = sp.rdbwsensitivity(df_rd, y='gpa', x='test_score', c=60, n_grid=20)
fig = plt.gcf()
sp.interactive(fig)

FigureEditor(fig=<Figure size 1500x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5305437200: <ArtistRole.LABEL: 5>, 5308659280: <ArtistRole.LABEL: 5>, 5308004432: <ArtistRole.LABEL: 5>, 5308093776: <ArtistRole.SPINE: 8>, 5308339472: <ArtistRole.SPINE: 8>, 5308021840: <ArtistRole.SPINE: 8>, 5308015312: <ArtistRole.SPINE: 8>, 5308256080: <ArtistRole.LEGEND: 7>, 5308011344: <ArtistRole.DATA: 1>, 5307934736: <ArtistRole.REFERENCE: 4>, 5308124240: <ArtistRole.REFERENCE: 4>, 5308337360: <ArtistRole.REFERENCE: 4>, 5308714832: <ArtistRole.REFERENCE: 4>, 5308346640: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'Bandwidth Sensitivity', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Bandwidth', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'RD Estimate', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(1.274459825), np.float64(6.222362675)), 'ax.ylim': (np.

---
## 7. Binscatter

Non-parametric conditional mean plots — the workhorse of applied micro visualization.

---
## 7. 分箱散点图 (Binscatter)

非参数条件均值图 —— 应用微观经济学可视化的常用工具。

In [30]:
# 分箱散点图：工资 vs 教育年限（来自 IV 数据），带置信区间
# Binscatter: wage vs education (from IV data) with CI bands
fig, ax, _ = sp.binscatter(
    df_iv, y='wage', x='education',
    n_bins=30, ci=True, ci_level=0.95,
    fit='linear',
    figsize=(9, 6),
)
ax.set_title('Binscatter: Wage vs Education', fontsize=13)
ax.set_xlabel('Years of Education', fontsize=11)
ax.set_ylabel('Hourly Wage ($)', fontsize=11)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1350x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5309637712: <ArtistRole.LABEL: 5>, 5358997712: <ArtistRole.LABEL: 5>, 5358994000: <ArtistRole.LABEL: 5>, 5304454480: <ArtistRole.SPINE: 8>, 5309492688: <ArtistRole.SPINE: 8>, 5306108560: <ArtistRole.SPINE: 8>, 5307901008: <ArtistRole.SPINE: 8>, 5359150544: <ArtistRole.REFERENCE: 4>, 5307857808: <ArtistRole.REFERENCE: 4>, 5359224016: <ArtistRole.DATA: 1>, 5307936976: <ArtistRole.DATA: 1>, 5304318544: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'Binscatter: Wage vs Education', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Years of Education', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'Hourly Wage ($)', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(7.230837929080427), np.float64(16.794586435945824)), 'ax.ylim': (np.float64(17.30389032141836), np.float64(27.3748

In [31]:
# 分箱散点图：工资 vs 教育年限，控制工作经验（二次拟合）
# Binscatter: wage vs education controlling for experience (quadratic fit)
fig, ax, _ = sp.binscatter(
    df_iv, y='wage', x='education',
    controls=['experience'],
    n_bins=25, ci=True,
    fit='quadratic',
    figsize=(9, 6),
)
ax.set_title('Binscatter: Wage vs Education\n(controlling for experience, quadratic fit)', fontsize=12)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1350x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5359723280: <ArtistRole.LABEL: 5>, 5535479376: <ArtistRole.LABEL: 5>, 5359786448: <ArtistRole.LABEL: 5>, 5535113360: <ArtistRole.SPINE: 8>, 5359542608: <ArtistRole.SPINE: 8>, 5309230032: <ArtistRole.SPINE: 8>, 5309930576: <ArtistRole.SPINE: 8>, 5535478928: <ArtistRole.REFERENCE: 4>, 5359467856: <ArtistRole.REFERENCE: 4>, 5359453264: <ArtistRole.DATA: 1>, 5359260880: <ArtistRole.DATA: 1>, 5308647568: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'Binscatter: Wage vs Education\n(controlling for experience, quadratic fit)', 'ax.title.fontsize': 12.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'education', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.text': 'wage', 'ax.ylabel.fontsize': 11.0, 'ax.ylabel.color': 'black', 'ax.xlim': (np.float64(7.391837015178976), np.float64(16.640096996074455)), 'ax.ylim': (np.float64(17.6017644417

In [32]:
# 按组分箱散点图：收入 vs 年龄，按处理状态分组
# Binscatter by group: earnings vs age, split by treatment
fig, ax, _ = sp.binscatter(
    df_match, y='earnings', x='age',
    by='treatment',
    n_bins=20, ci=True,
    fit='linear',
    figsize=(9, 6),
)
ax.set_title('Binscatter: Earnings vs Age by Treatment Status', fontsize=13)
ax.set_xlabel('Age', fontsize=11)
ax.set_ylabel('Earnings ($)', fontsize=11)
sp.interactive(fig)

FigureEditor(fig=<Figure size 1350x900 with 1 Axes>, protect_data=True, edits=[], artist_roles={5536571984: <ArtistRole.LABEL: 5>, 5536872016: <ArtistRole.LABEL: 5>, 5536331472: <ArtistRole.LABEL: 5>, 5536866448: <ArtistRole.SPINE: 8>, 5535811856: <ArtistRole.SPINE: 8>, 5535529040: <ArtistRole.SPINE: 8>, 5536035792: <ArtistRole.SPINE: 8>, 5536198608: <ArtistRole.LEGEND: 7>, 5536866704: <ArtistRole.REFERENCE: 4>, 5359549008: <ArtistRole.REFERENCE: 4>, 5535114960: <ArtistRole.DATA: 1>, 5535697872: <ArtistRole.REFERENCE: 4>, 5359374736: <ArtistRole.REFERENCE: 4>, 5536905936: <ArtistRole.DATA: 1>, 5359802640: <ArtistRole.DATA: 1>, 5537002640: <ArtistRole.CI: 3>, 5536697040: <ArtistRole.DATA: 1>, 5536699792: <ArtistRole.CI: 3>}, _original_state={'ax.title.text': 'Binscatter: Earnings vs Age by Treatment Status', 'ax.title.fontsize': 13.0, 'ax.title.color': 'black', 'ax.title.fontweight': 'normal', 'ax.xlabel.text': 'Age', 'ax.xlabel.fontsize': 11.0, 'ax.xlabel.color': 'black', 'ax.ylabel.te

---
## 8. Multi-Panel Comparison

Combining multiple methods in a single figure for publication.

---
## 8. 多面板对比图

将多种因果推断方法整合到单张图中，适用于学术发表。

In [33]:
# 多面板图：4 种因果推断方法并排展示
# Multi-panel figure: 4 causal methods side by side
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: DID parallel trends
sp.did_plot(
    df_did, y='outcome', time='time', treat='treated',
    treat_time=6, ax=axes[0, 0],
    title='(A) DID: Parallel Trends',
)

# Panel B: RD design
sp.rdplot(
    df_rd, y='gpa', x='test_score', c=60,
    nbins=30, ax=axes[0, 1],
    title='(B) RDD: Scholarship → GPA',
)

# Panel C: Synthetic control trajectory
sp.synthplot(sc_result, type='trajectory',
             ax=axes[1, 0],
             title='(C) SC: Treated vs Synthetic')

# Panel D: Binscatter wage vs education
sp.binscatter(
    df_iv, y='wage', x='education',
    n_bins=20, ci=True, fit='linear',
    ax=axes[1, 1],
)
axes[1, 1].set_title('(D) Binscatter: Wage vs Education', fontsize=11)

fig.suptitle('StatsPAI: Multi-Method Causal Inference', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
sp.interactive(fig)

FigureEditor(fig=<Figure size 2100x1500 with 4 Axes>, protect_data=True, edits=[], artist_roles={5537768016: <ArtistRole.LABEL: 5>, 5536642640: <ArtistRole.LABEL: 5>, 5538078224: <ArtistRole.LABEL: 5>, 5536321616: <ArtistRole.SPINE: 8>, 5537344144: <ArtistRole.SPINE: 8>, 5537970128: <ArtistRole.SPINE: 8>, 5537962576: <ArtistRole.SPINE: 8>, 5538116240: <ArtistRole.LEGEND: 7>, 5538541072: <ArtistRole.DATA: 1>, 5538904336: <ArtistRole.REFERENCE: 4>, 5538961296: <ArtistRole.DATA: 1>, 5538313360: <ArtistRole.REFERENCE: 4>, 5538963792: <ArtistRole.ANNOTATION: 6>, 5537666832: <ArtistRole.ANNOTATION: 6>, 5538162960: <ArtistRole.ANNOTATION: 6>, 5537500368: <ArtistRole.LABEL: 5>, 5537672976: <ArtistRole.LABEL: 5>, 5537675600: <ArtistRole.LABEL: 5>, 5537667536: <ArtistRole.SPINE: 8>, 5537669008: <ArtistRole.SPINE: 8>, 5537670096: <ArtistRole.SPINE: 8>, 5537671184: <ArtistRole.SPINE: 8>, 5538588880: <ArtistRole.DATA: 1>, 5538966800: <ArtistRole.DATA: 1>, 5537295440: <ArtistRole.REFERENCE: 4>, 5551

---
## Summary

| # | Method | Key Function | Plot Functions | Interactive |
|---|--------|-------------|----------------|-------------|
| 1 | **DID** | `sp.did(...)` | `sp.did_plot()`, `sp.event_study_plot()` | `sp.interactive(fig)` |
| 2 | **FE** | `sp.panel(..., method='fe')` | `sp.coefplot()` | `sp.interactive(fig)` |
| 3 | **IV** | `sp.iv("y ~ (endog ~ inst) + exog")` | `sp.coefplot()` | `sp.interactive(fig)` |
| 4 | **Matching** | `sp.match(...)` | `sp.balanceplot()`, `sp.psplot()` | `sp.interactive(fig)` |
| 5 | **SC** | `sp.synth(...)` | `sp.synthplot(..., type=)` | `sp.interactive(fig)` |
| 6 | **RDD** | `sp.rdrobust(...)` | `sp.rdplot()`, `sp.rdplotdensity()` | `sp.interactive(fig)` |
| 7 | **Binscatter** | `sp.binscatter(...)` | built-in | `sp.interactive(fig)` |

### Interactive Editing Tips

- **New in this version**: DPI slider, figure size presets, background color, tick rotation, annotations, grid style, copy-to-clipboard
- Use `sp.get_code(fig)` to extract reproducible code after editing
- All edits are cosmetic only — statistical data is locked and protected

---
## 总结

| # | 方法 | 核心函数 | 绘图函数 | 交互式 |
|---|------|---------|---------|--------|
| 1 | **双重差分法** | `sp.did(...)` | `sp.did_plot()`, `sp.event_study_plot()` | `sp.interactive(fig)` |
| 2 | **固定效应** | `sp.panel(..., method='fe')` | `sp.coefplot()` | `sp.interactive(fig)` |
| 3 | **工具变量** | `sp.iv("y ~ (endog ~ inst) + exog")` | `sp.coefplot()` | `sp.interactive(fig)` |
| 4 | **匹配法** | `sp.match(...)` | `sp.balanceplot()`, `sp.psplot()` | `sp.interactive(fig)` |
| 5 | **合成控制** | `sp.synth(...)` | `sp.synthplot(..., type=)` | `sp.interactive(fig)` |
| 6 | **断点回归** | `sp.rdrobust(...)` | `sp.rdplot()`, `sp.rdplotdensity()` | `sp.interactive(fig)` |
| 7 | **分箱散点图** | `sp.binscatter(...)` | 内置 | `sp.interactive(fig)` |

### 交互式编辑提示

- **本版新增**：DPI 滑块、图形大小预设、背景颜色、刻度旋转、注释、网格样式、复制到剪贴板
- 使用 `sp.get_code(fig)` 在编辑后提取可复现代码
- 所有编辑仅影响外观 —— 统计数据被锁定保护，不会被修改